Generate Forecast — Live Production Data
Loads gold data, trains final weekly (LightGBM) and monthly (Prophet, flat growth) models on all available history, and saves a single combined forecast output.

**Input**: gold/erp/battery/phase1_overall_weekly_live.parquet, phase1_overall_monthly_live.parquet
**Output**: gold/forecasts/overall_forecast_latest.json

In [0]:
%run ./_local_config

In [0]:
%pip install lightgbm prophet

In [0]:
dbutils.library.restartPython()

In [0]:
%run ./_local_config

In [0]:
import sys
sys.path.append("/Workspace/Users/venura-it@brownsgroup.com/Exide sales/Exide-Sales-Forecast")

from src.io.storage import get_blob_service, read_gold
import pandas as pd
import datetime
import lightgbm as lgb
from prophet import Prophet

blob_service = get_blob_service(storage_account_name, storage_account_key)

Weekly forecast

In [0]:
gold_weekly = read_gold(blob_service, "live/phase1_overall_weekly_live.parquet")
gold_weekly["week_start"] = pd.to_datetime(gold_weekly["week_start"])

feature_cols = ["week_of_year", "month", "contains_month_end", "lag_4w", "rolling_avg_4w"]
target_col = "total_units_sold"

model_data = gold_weekly.dropna(subset=feature_cols + [target_col]).copy()

final_weekly_model = lgb.LGBMRegressor(n_estimators=200, learning_rate=0.05, max_depth=6)
final_weekly_model.fit(model_data[feature_cols], model_data[target_col])

last_week_start = gold_weekly["week_start"].max()
next_week_start = last_week_start + pd.Timedelta(weeks=1)
next_week_end = next_week_start + pd.Timedelta(days=6)

lag_4w_value = gold_weekly[gold_weekly["week_start"] == next_week_start - pd.Timedelta(weeks=4)]["total_units_sold"].values
lag_4w_value = lag_4w_value[0] if len(lag_4w_value) > 0 else None

next_week_features = pd.DataFrame([{
    "week_of_year": next_week_start.isocalendar()[1],
    "month": next_week_start.month,
    "contains_month_end": int(next_week_start.month != next_week_end.month),
    "lag_4w": lag_4w_value,
    "rolling_avg_4w": gold_weekly["total_units_sold"].tail(4).mean(),
}])

weekly_prediction = final_weekly_model.predict(next_week_features[feature_cols])[0]
print(f"Weekly forecast — week of {next_week_start.date()}: {weekly_prediction:.0f}")

Monthly forecast

In [0]:
gold_monthly = read_gold(blob_service, "live/phase1_overall_monthly_live.parquet")
gold_monthly["month_start"] = pd.to_datetime(gold_monthly["month_start"])

monthly_prophet_df = gold_monthly[["month_start", "total_units_sold"]].rename(
    columns={"month_start": "ds", "total_units_sold": "y"}
)

m_final = Prophet(yearly_seasonality=True, weekly_seasonality=False, growth='flat')
m_final.fit(monthly_prophet_df)

future_final = m_final.make_future_dataframe(periods=3, freq="MS")
forecast_final = m_final.predict(future_final)
forecast_final[["yhat", "yhat_lower", "yhat_upper"]] = forecast_final[["yhat", "yhat_lower", "yhat_upper"]].clip(lower=0)

monthly_forecast = forecast_final[["ds", "yhat", "yhat_lower", "yhat_upper"]].tail(3)
print(monthly_forecast)

Combine and save

In [0]:
import json

forecast_output = {
    "generated_at": datetime.datetime.now().isoformat(),
    "weekly": {
        "week_start": next_week_start.strftime("%Y-%m-%d"),
        "predicted_units": round(float(weekly_prediction))
    },
    "monthly": [
        {
            "month_start": row["ds"].strftime("%Y-%m-%d"),
            "predicted_units": round(float(row["yhat"])),
            "lower_bound": round(float(row["yhat_lower"])),
            "upper_bound": round(float(row["yhat_upper"]))
        }
        for _, row in monthly_forecast.iterrows()
    ]
}

print(json.dumps(forecast_output, indent=2))

blob_client = blob_service.get_blob_client(container="gold", blob="live/forecasts/overall_forecast_latest.json")
blob_client.upload_blob(json.dumps(forecast_output, indent=2), overwrite=True)
print("Saved to gold/live/forecasts/overall_forecast_latest.json")